# Mercury — Parakeet + CosyVoice2 Fine-Tuning Notebook

Run this on Kaggle (Settings → Accelerator → GPU T4 x2 or P100).

**What this notebook does:**
1. Installs NeMo (ASR) and CosyVoice2 dependencies
2. Downloads pretrained `nvidia/parakeet-tdt-0.6b-v2` (ASR)
3. Downloads pretrained `iic/CosyVoice2-0.5B` (TTS)
4. Smoke-tests both models (inference only, no training yet)
5. Downloads fine-tuning data: **MedDialog-Audio** (synthetic, ~165GB) + **PriMock57** (real, small)
6. Leaves fine-tuning scaffolds ready to run once manifests are built

> Both models are permissively licensed (Parakeet: CC-BY-4.0, CosyVoice2: Apache 2.0) — fine for a student/non-commercial project.
> MedDialog-Audio is CC BY-NC 4.0 — non-commercial only, matches this project.
> **Kaggle disk note:** MedDialog-Audio is ~165GB. Kaggle's working directory has a 20GB persistent-output limit and ~73GB scratch disk depending on instance — do NOT try to pull the full set. Use the streaming/subset approach in section 5 instead of a full download.

## 0. Environment check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Install dependencies
This takes a few minutes on first run.

In [ ]:
%%capture
!pip install -q -U pip

!pip install -q "nemo_toolkit[asr]"

!pip install -q modelscope funasr datasets huggingface_hub
!git clone --depth 1 https://github.com/FunAudioLLM/CosyVoice.git /kaggle/working/CosyVoice
!pip install -q -r /kaggle/working/CosyVoice/requirements.txt

## 2. Download + load Parakeet-TDT (ASR)

In [ ]:
import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v2")
print('Parakeet loaded.', sum(p.numel() for p in asr_model.parameters()) / 1e6, 'M params')

In [ ]:
# Smoke test — replace with a real wav path once you have sample audio uploaded as a Kaggle dataset
# output = asr_model.transcribe(['/kaggle/input/sample-audio/test.wav'])
# print(output[0].text)

## 3. Download + load CosyVoice2 (TTS)

In [ ]:
from modelscope import snapshot_download

cosyvoice_dir = snapshot_download('iic/CosyVoice2-0.5B', cache_dir='/kaggle/working/models')
print('CosyVoice2 weights at:', cosyvoice_dir)

In [ ]:
import sys
sys.path.append('/kaggle/working/CosyVoice')
sys.path.append('/kaggle/working/CosyVoice/third_party/Matcha-TTS')

from cosyvoice.cli.cosyvoice import CosyVoice2

tts_model = CosyVoice2(cosyvoice_dir, load_jit=False, load_trt=False, fp16=False)
print('CosyVoice2 loaded.')

In [ ]:
# Smoke test — zero-shot needs a reference prompt wav; sft mode uses built-in speakers
# print(tts_model.list_available_spks())
# for out in tts_model.inference_sft('This is a Mercury fine-tuning test.', tts_model.list_available_spks()[0]):
#     import torchaudio
#     torchaudio.save('/kaggle/working/test_out.wav', out['tts_speech'], tts_model.sample_rate)

## 4. PriMock57 (real medical consultation audio)

Small (57 mock consultations), free, non-commercial research license. Good for validation / a real-audio slice alongside the synthetic set.

In [ ]:
!git clone --depth 1 https://github.com/babylonhealth/primock57.git /kaggle/working/data/primock57
!find /kaggle/working/data/primock57 -maxdepth 2 -type d

## 5. MedDialog-Audio (synthetic medical speech)

Full dataset is ~165GB (147,476 files) — too big for Kaggle's disk quota. Stream a bounded subset instead
of downloading everything; adjust `N_SAMPLES` to whatever fits your fine-tuning plan.

License: CC BY-NC 4.0 — non-commercial, matches this project.

In [ ]:
import os
from datasets import load_dataset
import soundfile as sf

N_SAMPLES = 5000
OUT_DIR = '/kaggle/working/data/meddialog_audio'
os.makedirs(OUT_DIR, exist_ok=True)

ds = load_dataset('aline-gassenn/MedDialog-Audio', split='train', streaming=True)

manifest_rows = []
for i, row in enumerate(ds):
    if i >= N_SAMPLES:
        break
    audio = row['audio']
    text = row.get('text') or row.get('transcript') or ''
    wav_path = os.path.join(OUT_DIR, f'{i:06d}.wav')
    sf.write(wav_path, audio['array'], audio['sampling_rate'])
    duration = len(audio['array']) / audio['sampling_rate']
    manifest_rows.append({'audio_filepath': wav_path, 'text': text, 'duration': duration})

print(f'Downloaded {len(manifest_rows)} samples to {OUT_DIR}')

## 6. Build NeMo manifests (train/val split)

Combines PriMock57 (real) + the MedDialog-Audio subset (synthetic) into NeMo-format JSONL manifests.
PriMock57 ships transcripts as separate text files — adjust the parsing block below once you've inspected
its actual directory layout (`interview1/`, `interview1.wav`, etc. — verify column names).

In [ ]:
import json, random

primock_rows = []
# for wav in glob.glob('/kaggle/working/data/primock57/**/*.wav', recursive=True):
#     txt = wav.replace('.wav', '.txt')
#     if os.path.exists(txt):
#         text = open(txt).read().strip()
#         duration = sf.info(wav).duration
#         primock_rows.append({'audio_filepath': wav, 'text': text, 'duration': duration})

all_rows = manifest_rows + primock_rows
random.seed(42)
random.shuffle(all_rows)

split = int(len(all_rows) * 0.9)
train_rows, val_rows = all_rows[:split], all_rows[split:]

with open('/kaggle/working/train_manifest.json', 'w') as f:
    for r in train_rows:
        f.write(json.dumps(r) + '\n')
with open('/kaggle/working/val_manifest.json', 'w') as f:
    for r in val_rows:
        f.write(json.dumps(r) + '\n')

print(f'train: {len(train_rows)} rows, val: {len(val_rows)} rows')

## 7. Fine-tuning — Parakeet (ASR)

Adapter/LoRA fine-tuning recommended given Kaggle's GPU quota (30hrs/week) rather than a full fine-tune.

In [ ]:
# --- uncomment once section 6 manifests are populated ---
# asr_model.setup_training_data(train_data_config={
#     'manifest_filepath': '/kaggle/working/train_manifest.json',
#     'batch_size': 8,
# })
# asr_model.setup_validation_data(val_data_config={
#     'manifest_filepath': '/kaggle/working/val_manifest.json',
#     'batch_size': 8,
# })
#
# import pytorch_lightning as pl
# trainer = pl.Trainer(accelerator='gpu', devices=1, max_epochs=5, precision='bf16-mixed',
#                       default_root_dir='/kaggle/working/checkpoints')
# trainer.fit(asr_model)

## 8. Fine-tuning — CosyVoice2 (TTS)

CosyVoice2 fine-tuning uses its own training scripts under `CosyVoice/examples/`.

TODO:
- [ ] Format the manifests from section 6 per CosyVoice's `examples/libritts/cosyvoice2/` recipe (wav + text + speaker id)
- [ ] Run `CosyVoice/examples/.../run.sh` stages, or adapt the training script directly for a single medical narrator voice
- [ ] Save fine-tuned checkpoint to `/kaggle/working/models/cosyvoice2-medical`

In [ ]:
# --- placeholder ---
# !bash /kaggle/working/CosyVoice/examples/libritts/cosyvoice2/run.sh --stage 0 --stop_stage 3